# 04 — YOLOv8s Fault Injection

INT8/INT4 weight fault injection and preliminary propagation analysis.

**Scope:** YOLOv8s; single-bit faults; synthetic input; pre-NMS outputs.

**Limitations:** No COCO mAP or real-image fault evaluation yet.

## 1. Environment and model integrity

Load verified YOLOv8s weights and confirm the baseline SHA256.

In [ ]:
# YOLOV8S_MODEL_INTEGRITY_LOADER_V1
# Verify the original model checkpoint before loading it on CPU.

from pathlib import Path
import hashlib
from ultralytics import YOLO

project_root = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
model_path = project_root / "models/yolov8s.pt"

expected_size = 22588772
expected_sha256 = (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
)

assert model_path.is_file(), "YOLOv8s checkpoint missing"
assert model_path.stat().st_size == expected_size, "Model size mismatch"

digest = hashlib.sha256()
with model_path.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(chunk)

assert digest.hexdigest() == expected_sha256, "Model SHA256 mismatch"

yolo = YOLO(str(model_path))
model = yolo.model.cpu().eval()

assert len(yolo.names) == 80, "Unexpected class count"

print("Model: YOLOv8s")
print("Classes:", len(yolo.names))
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device: CPU")
print("SHA256: VERIFIED")
print("YOLOV8S MODEL INTEGRITY LOADER: PASS")


## 2. INT8 weight fault injection

Import the tested INT8 module and document boundary tests.

In [ ]:
# INT8_RESEARCH_MODULE_TEST_V1
import sys
from pathlib import Path
import torch

src_dir = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/src"
)
assert (src_dir / "int8_fault_injection.py").is_file()

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from int8_fault_injection import flip_int8_bit

original = torch.tensor([4], dtype=torch.int8)
faulty, record = flip_int8_bit(original, (0,), 7)

assert original.item() == 4
assert faulty.item() == -124
assert record["bit_position"] == 7

print("INT8 SINGLE-BIT MODULE TEST: PASS")


## 3. INT4 weight fault injection

Import the tested logical INT4 module and document exhaustive tests.

In [ ]:
# INT4_EXHAUSTIVE_TEST_V1
import sys
from pathlib import Path

src_dir = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/src"
)
assert (src_dir / "int4_fault_injection.py").is_file()

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from int4_fault_injection import flip_int4_bit

test_count = 0

for value in range(-8, 8):
    for bit in range(4):
        faulty, record = flip_int4_bit(value, bit)

        expected_bits = (value & 0xF) ^ (1 << bit)
        expected = (
            expected_bits
            if expected_bits < 8
            else expected_bits - 16
        )

        assert faulty == expected
        assert record["bit_position"] == bit

        test_count += 1

assert test_count == 64

print("INT4 EXHAUSTIVE TESTS:", test_count)
print("INT4 EXHAUSTIVE TEST: PASS")


## 4. INT4 model preparation

Construct quantized and single-fault model copies.

In [ ]:
# INT4_MODEL_PREPARATION_V1
import copy
import torch
from int4_fault_injection import flip_int4_tensor_bit

# Requires a previously loaded, verified YOLOv8s PyTorch model.
assert "model" in globals(), "Load verified YOLOv8s model first"
assert isinstance(model, torch.nn.Module)

conv = model.model[0].conv
w = conv.weight.detach()

assert tuple(w.shape) == (32, 3, 3, 3)
assert w.dtype == torch.float32

# Symmetric per-tensor INT4 quantization, logical values [-7, 7].
scale = w.abs().max() / 7
assert scale.item() > 0

q_int4 = torch.round(w / scale).clamp(-7, 7).to(torch.int8)

index = (0, 0, 0, 0)
faulty_q, fault_record = flip_int4_tensor_bit(
    q_int4, index, 3
)

quantized_model = copy.deepcopy(model)
faulty_model = copy.deepcopy(model)

with torch.no_grad():
    quantized_model.model[0].conv.weight.copy_(
        q_int4.float() * scale
    )
    faulty_model.model[0].conv.weight.copy_(
        faulty_q.float() * scale
    )

quantized_model.eval().cpu()
faulty_model.eval().cpu()

assert torch.equal(
    q_int4,
    torch.round(w / scale).clamp(-7, 7).to(torch.int8)
)
assert int((q_int4 != faulty_q).sum().item()) == 1

print("Fault position:", index)
print("Fault bit:", fault_record["bit_position"])
print("Quantization scale:", scale)
print("INT4 MODEL PREPARATION: PASS")


## 5. Synthetic-input inference

Compare clean-quantized and faulty outputs before NMS.

In [ ]:
# INT4_SYNTHETIC_INFERENCE_V1
import torch

assert "quantized_model" in globals()
assert "faulty_model" in globals()

quantized_model.eval().cpu()
faulty_model.eval().cpu()

# Deterministic synthetic input; not a COCO image.
torch.manual_seed(42)
x = torch.rand(1, 3, 640, 640)

with torch.inference_mode():
    output_q = quantized_model(x)
    output_f = faulty_model(x)

# YOLOv8 detection models may return tuples.
pred_q = output_q[0] if isinstance(output_q, tuple) else output_q
pred_f = output_f[0] if isinstance(output_f, tuple) else output_f

assert pred_q.shape == pred_f.shape == (1, 84, 8400)
assert torch.isfinite(pred_q).all()
assert torch.isfinite(pred_f).all()

difference = (pred_q - pred_f).abs()

print("Raw output shape:", tuple(pred_q.shape))
print("Mean absolute difference:", difference.mean().item())
print("Maximum absolute difference:", difference.max().item())
print("INT4 SYNTHETIC PRE-NMS INFERENCE: PASS")


## 6. Bounding-box and class-score analysis

Analyze raw output differences, center shifts and matched-box IoU.

In [ ]:
# INT4_BOX_CLASS_ANALYSIS_V1
import torch

assert "pred_q" in globals()
assert "pred_f" in globals()
assert pred_q.shape == pred_f.shape == (1, 84, 8400)

# YOLOv8s raw detection output:
# 4 bounding-box coordinates + 80 class scores.
box_diff = (pred_q[:, :4, :] - pred_f[:, :4, :]).abs()
class_diff = (pred_q[:, 4:, :] - pred_f[:, 4:, :]).abs()

def summarize_diff(name, values):
    assert torch.isfinite(values).all()
    print("\n" + name)
    print("Elements:", values.numel())
    print("Mean absolute difference:", values.mean().item())
    print("Maximum absolute difference:", values.max().item())
    print(
        "99th percentile:",
        torch.quantile(values.flatten(), 0.99).item()
    )

summarize_diff("Bounding-box coordinates", box_diff)
summarize_diff("Class scores", class_diff)

assert box_diff.numel() == 4 * 8400
assert class_diff.numel() == 80 * 8400

print("\nINT4 BOX-CLASS ANALYSIS: PASS")


In [ ]:
# INT4_CENTER_IOU_ANALYSIS_V1
import torch

assert pred_q.shape == pred_f.shape == (1, 84, 8400)

# Raw YOLOv8 boxes: (center_x, center_y, width, height).
boxes_q = pred_q[:, :4, :]
boxes_f = pred_f[:, :4, :]

centers_q = boxes_q[:, :2, :]
centers_f = boxes_f[:, :2, :]

center_shift = torch.linalg.vector_norm(
    centers_q - centers_f, dim=1
)

def xywh_to_xyxy(boxes):
    xy = boxes[:, :2, :]
    half_wh = boxes[:, 2:4, :].clamp(min=0) / 2
    return torch.cat((xy - half_wh, xy + half_wh), dim=1)

a = xywh_to_xyxy(boxes_q)
b = xywh_to_xyxy(boxes_f)

top_left = torch.maximum(a[:, :2, :], b[:, :2, :])
bottom_right = torch.minimum(a[:, 2:, :], b[:, 2:, :])

intersection_wh = (bottom_right - top_left).clamp(min=0)
intersection = intersection_wh[:, 0, :] * intersection_wh[:, 1, :]

area_a = (
    (a[:, 2, :] - a[:, 0, :]).clamp(min=0)
    * (a[:, 3, :] - a[:, 1, :]).clamp(min=0)
)
area_b = (
    (b[:, 2, :] - b[:, 0, :]).clamp(min=0)
    * (b[:, 3, :] - b[:, 1, :]).clamp(min=0)
)

union = area_a + area_b - intersection
iou = torch.where(
    union > 0,
    intersection / union.clamp(min=1e-12),
    torch.zeros_like(union)
)

assert center_shift.shape == (1, 8400)
assert iou.shape == (1, 8400)
assert torch.isfinite(center_shift).all()
assert torch.isfinite(iou).all()
assert ((iou >= 0) & (iou <= 1)).all()

print("Mean center shift (pixels):", center_shift.mean().item())
print("Median center shift (pixels):", center_shift.median().item())
print("Boxes with shift > 1 pixel:", int((center_shift > 1).sum()))
print("Mean matched raw-box IoU:", iou.mean().item())
print("Boxes with IoU < 0.9:", int((iou < 0.9).sum()))
print("INT4 CENTER SHIFT / IOU ANALYSIS: PASS")


## 7. Results and limitations

Reference existing JSON artifacts without overwriting them.

### INT4 numerical reproducibility note

The historical experiment used FP32 tensor division:

`scale = w.abs().max() / 7`

The earlier research draft used Python scalar division:

`scale = w.abs().max().item() / 7`

This difference explains the observed numerical discrepancy:

- Historical FP32 tensor scale: `0.0695452019572258`
- Earlier Python scalar scale: `0.06954520089285714`
- Absolute difference: `1.0643686626510274e-09`
- Relative difference: `1.5304703023303804e-08`

The historical expression was confirmed in cell 42 of the
original notebook archive.

The research draft has been corrected to use FP32 tensor
division. An independent CPU calculation confirmed an exact
numerical match with the recovered historical scale.

The fault configuration also matches the historical record:

- Weight index: `(0, 0, 0, 0)`
- Bit position: `3`
- Logical INT4 transition: `0 -> -8`

**Reproducibility status:** The quantization-scale calculation
and fault configuration have been verified. The corrected
model-preparation cell has not yet been executed again, and
historical inference outputs have not been reproduced.

This experiment simulates INT4-quantized weights within FP32
PyTorch models. It does not establish native INT4 hardware
performance, COCO detection accuracy, or fault tolerance
under real-image inference.


### Historical INT4 synthetic fault-propagation findings

**Source:** Recovered JSON reconstructed from outputs in the original
experiment notebook archive. These are historical results, not newly
reproduced inference measurements.

**Experimental scope**
- Model: YOLOv8s
- Input: one seeded synthetic image, not a COCO validation image
- Fault: one INT4 weight bit flip in the first convolutional layer
- Weight index: `(0, 0, 0, 0)`
- Bit position: `3`
- Logical INT4 transition: `0 -> -8`
- Analysis: paired raw predictions before NMS
- Output shape: `(1, 84, 8400)`

**Recorded observations**
- Mean paired raw-box IoU: `0.941421330`
- Median paired raw-box IoU: `0.953006446`
- Minimum paired raw-box IoU: `0.408197403`
- Mean center displacement: `1.075070 px`
- Maximum center displacement: `15.084360 px`
- Paired boxes with IoU below 0.50: `1`
- Paired boxes with IoU below 0.75: `51`
- Paired boxes with IoU below 0.90: `1104`
  of 8400 (13.1429%)

**Interpretation and limitations**

The injected weight fault affected raw bounding-box predictions
in this single synthetic-input experiment. Paired IoU compares
same-index predictions, not ground-truth detection accuracy.

These results do not measure COCO mAP, post-NMS detections,
real-image robustness, hardware INT4 execution, or the
effectiveness of any fault-tolerance mitigation.

Internal consistency checks of the recovered JSON passed;
they do not independently verify the original inference.


In [ ]:
# INT4_EXISTING_RESULTS_READER_V1
# Read-only access to results recovered from archived experiment outputs.

from pathlib import Path
import json

project_root = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
result_path = (
    project_root
    / "notebook_backups"
    / "int4_single_fault_synthetic_recovered_v1.json"
)

assert result_path.is_file(), "Recovered INT4 results not found"

with result_path.open("r", encoding="utf-8") as f:
    int4_result = json.load(f)

assert int4_result["model"] == "YOLOv8s"
assert int4_result["input"]["type"] == "synthetic_random"
assert int4_result["input"]["seed"] == 42
assert int4_result["fault"]["bit_position"] == 3

print("Experiment:", int4_result["experiment"])
print("Fault:", int4_result["fault"])
print("Mean paired IoU:", int4_result["output"]["mean_paired_iou"])
print("Mean center shift (px):",
      int4_result["output"]["mean_center_shift_px"])
print("Source: recovered JSON from archived experiment outputs")
print("INT4 EXISTING RESULTS READER: PASS")
